In [87]:
import numpy as np 
import gmsh
import vtk
import math
import os

In [88]:
# just to ensure nothing breaks because gmsh can 
# and I cannot trust it anymoooore T~T 

gmsh.initialize()
gmsh.finalize()


In [89]:
# Класс расчётной сетки
class CalcMesh:

    # Конструктор сетки, полученной из stl-файла
    def __init__(self, nodes_coords, tetrs_points):
        self.nodes = np.array([nodes_coords[0::3],nodes_coords[1::3],nodes_coords[2::3]])
        self.original_nodes = self.nodes.copy()
        self.N = self.nodes.shape[1]


        self.smth = np.power(self.nodes[0, :], 2) + np.power(self.nodes[1, :], 2)
        self.tetrs = np.array([tetrs_points[0::4],tetrs_points[1::4],tetrs_points[2::4],tetrs_points[3::4]])
        self.tetrs -= 1

        self.velocity = np.zeros((3, self.N), dtype=np.double)
        self.velocity[2] = 0.2

        self.split = np.zeros(3)


    def move(self, tau):
        self.split += self.velocity.mean(axis=1) * tau


    # method responsible for tilting around X axis and then rotating around Z 
    def rotate(self, angle_deg, axis='z', tilt_deg='None'):
        theta = np.radians(angle_deg)
        rotated = self.original_nodes.copy()

        # rotation relatively to X
        if tilt_deg != 'None':
            alpha = np.radians(tilt_deg)
            Rx = np.array([[1, 0, 0], [0, np.cos(alpha), np.sin(-alpha)], [0, np.sin(alpha), np.cos(alpha)]])
            rotated = Rx @ rotated # I keep forgeting that @ is matrix multiplication. I will leave that little shame of mine here and nobody would ever ready it anyway, r-right?

        # rotation relatively to Z
        Rz = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0,0,1]])
        rotated = Rz @ rotated

        self.nodes = rotated + self.split.reshape(3,1)


    # Метод отвечает за запись текущего состояния сетки в снапшот в формате VTK
    def snapshot(self, snap_number):
        unstructuredGrid = vtk.vtkUnstructuredGrid()
        points = vtk.vtkPoints()

        smth = vtk.vtkDoubleArray()
        smth.SetName("smth")

        vel = vtk.vtkDoubleArray()
        vel.SetNumberOfComponents(3)
        vel.SetName("vel")

        for i in range(0, self.N):
            points.InsertNextPoint(self.nodes[0,i], self.nodes[1,i], self.nodes[2,i])
            smth.InsertNextValue(self.smth[i])
            vel.InsertNextTuple((self.velocity[0,i], self.velocity[1,i], self.velocity[2,i]))



        # Грузим точки в сетку
        unstructuredGrid.SetPoints(points)

        # Присоединяем векторное и скалярное поля к точкам
        unstructuredGrid.GetPointData().AddArray(smth)
        unstructuredGrid.GetPointData().AddArray(vel)

        for i in range(0, self.tetrs.shape[1]):
            tetr = vtk.vtkTetra()
            for j in range(0, 4):
                tetr.GetPointIds().SetId(j, int(self.tetrs[j,i]))
            # unstructuredGrid.InsertNextCell(tetr.GetCellType(), tetr.GetPointIds())
            unstructuredGrid.InsertNextCell(tetr.GetCellType(), tetr.GetPointIds())

        writer = vtk.vtkXMLUnstructuredGridWriter()
        writer.SetInputDataObject(unstructuredGrid)
        writer.SetFileName(f"friend-step-{snap_number:03d}.vtu")
        writer.Write()



In [90]:
gmsh.initialize()

try:
    # path = os.path.dirname(os.path.abspath(__file__))
    gmsh.merge('friend.stl')
except:
    print("Could not load STL mesh: bye!")
    gmsh.finalize()
    exit(-1)


Info    : Reading 'friend.stl'...
Info    : 5264 facets in solid 0 Default
Info    : Done reading 'friend.stl'


In [ ]:
angle = 15  
gmsh.model.mesh.classifySurfaces(angle * math.pi/180., True, False, 180 * math.pi/180.)
gmsh.model.mesh.createGeometry()

surfaces = gmsh.model.getEntities(2)
surface_tags = [tag for (dim, tag) in surfaces]
loop = gmsh.model.geo.addSurfaceLoop(surface_tags)
vol = gmsh.model.geo.addVolume([loop])
gmsh.model.geo.synchronize()

field = gmsh.model.mesh.field.add("MathEval")
gmsh.model.mesh.field.setString(field, "F", "2")  
gmsh.model.mesh.field.setAsBackgroundMesh(field)

gmsh.model.mesh.generate(3)
gmsh.write('friend.msh')


In [92]:
nodeTags, nodesCoord, parametricCoord = gmsh.model.mesh.getNodes()


# И данные об элементах сетки тоже извлечём, нам среди них нужны только тетраэдры, которыми залит объём
GMSH_TETR_CODE = 4
tetrsNodesTags = None
elementTypes, elementTags, elementNodeTags = gmsh.model.mesh.getElements()
for i in range(0, len(elementTypes)):
    if elementTypes[i] != GMSH_TETR_CODE:
        continue
    tetrsNodesTags = elementNodeTags[i]

if tetrsNodesTags is None:
    print("Can not find tetra data. Exiting.")
    gmsh.finalize()
    exit(-2)


print("The model has %d nodes and %d tetrs" % (len(nodeTags), len(tetrsNodesTags) / 4))


The model has 15603 nodes and 67595 tetrs


In [93]:
tilt = 45.0

mesh = CalcMesh(nodesCoord, tetrsNodesTags)

for i in range(0, 99):
    angle = i*10.0
    mesh.rotate(angle, tilt_deg = tilt)
    mesh.move(1)
    mesh.snapshot(i)
gmsh.finalize()